# Marco Geoestadistico


In [81]:
import yarl 
import requests
from scripts.download import download
from scripts.config import DATA_FOLDERS, get_project_root
from zipfile import ZipFile
import os

raw_data = DATA_FOLDERS['raw'] / "inegi"
ext_data = DATA_FOLDERS['external'] / "inegi"

mexico_map = "https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/794551196649/mg_integrado_encuesta_intercensal_2025.zip"
MEXICO_URL = yarl.URL(mexico_map)
if not os.path.exists(get_project_root() / "docs" / "mexico_agro.zip"):
    check_mexicomap = requests.get(MEXICO_URL)
    if check_mexicomap.status_code==200:
            download(yarl.URL(MEXICO_URL), "Mexico.zip", ext_data)
            with ZipFile(ext_data / "Mexico.zip", "r") as mexico:
                mexico.extractall(ext_data)


In [82]:


url = MEXICO_URL
output = ext_data / "Mexico.zip"

chunk_size = 4 * 1024 * 1024  # 4 MB

downloaded = output.stat().st_size if output.exists() else 0

headers = {}

if downloaded:
    headers["Range"] = f"bytes={downloaded}-"

with requests.get(
    url,
    headers=headers,
    stream=True,
    timeout=(10, 120),
) as response:

    # 206 = servidor aceptó Range
    # 200 = descarga completa normal
    if downloaded and response.status_code != 206:
        print("El servidor no soportó reanudación.")
        downloaded = 0
        mode = "wb"
    else:
        mode = "ab" if downloaded else "wb"

    response.raise_for_status()

    remaining = int(response.headers.get("content-length", 0))
    total = downloaded + remaining

    with open(output, mode) as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if not chunk:
                continue

            f.write(chunk)
            downloaded += len(chunk)

            if total:
                print(
                    f"\r{downloaded / 1024**3:.2f} / "
                    f"{total / 1024**3:.2f} GB "
                    f"({100 * downloaded / total:.1f}%)",
                    end="",
                )

print("\nDescarga terminada.")

0.05 / 0.24 GB (19.3%)

KeyboardInterrupt: 

# Censo Agropecuario 2022

In [70]:
from scripts.config import DATA_FOLDERS
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from zipfile import ZipFile
import yarl
import os
from scripts.download import download

raw_inegi_path = DATA_FOLDERS['raw'] / "inegi"
ext_inegi_path = DATA_FOLDERS['external'] / "inegi"

inegi_ca = yarl.URL("https://www.inegi.org.mx/contenidos/programas/")
files = ["cagf/2007/datosabiertos/cagf2007_csv.zip"] + [f"ca/2022/datosabiertos/{file}" 
for file in ("ca_2022_superficie_csv.zip", "ca_2022_upagro_csv.zip", "ca_2022_upfores_csv.zip")]

for f in files:
    file_name = f.split("/")[-1]
    download(inegi_ca / f, file_name, raw_inegi_path)


https://www.inegi.org.mx/contenidos/programas/cagf/2007/datosabiertos/cagf2007_csv.zip cagf2007_csv.zip /home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi /home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi/cagf2007_csv.zip
El archivo ya fue descargado. Se encuentra en /home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi/cagf2007_csv.zip
https://www.inegi.org.mx/contenidos/programas/ca/2022/datosabiertos/ca_2022_superficie_csv.zip ca_2022_superficie_csv.zip /home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi /home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi/ca_2022_superficie_csv.zip
El archivo ya fue descargado. Se encuentra en /home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi/ca_2022_superficie_csv.zip
https://www.inegi.org.mx/contenidos/programas/ca/2022/datosabiertos/ca_2022_upagro_csv.zip ca_2022_upagro_csv.zip /home/

In [76]:
inegi_files= os.listdir(raw_inegi_path)

for igf in inegi_files:
    zipd = raw_inegi_path / igf
    fext = igf.split(".")[1]
    name = igf.split(".")[0].rstrip("_csv")
    print(zipd)
    with ZipFile(zipd, "r") as fz:
        fz.extractall(ext_inegi_path / f"{name}")

/home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi/cagf2007_csv.zip
/home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi/ca_2022_superficie_csv.zip
/home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi/ca_2022_upagro_csv.zip
/home/david/Documents/Repositories/academica/proyecto_agronomia/data/raw/inegi/ca_2022_upfores_csv.zip


In [25]:
response = requests.get(access_url)
print(access_url)
response.raise_for_status()

https://www.inegi.org.mx/programas/ca/2022
